# Reproducing the six- and seven-voter computations

This notebook reproduces the computer-assisted claims in Section 5 of the
paper in the order in which the mathematical objects arise. It first explains
the finite residual search, works through a concrete hole, constructs the
catalogue, and only then introduces the fact-specific checks.

The computation uses exact Z3 rational arithmetic for continuous feasibility
and exhaustive enumeration for integral committees. No floating-point
optimization is used.


## 1. The finite search space

For a family of candidate types $\mathcal F$, a fractional committee is

$$
x\in[0,1]^{\mathcal F},\qquad \sum_{R\in\mathcal F}x_R\le k,
$$

and voter $i$'s utility is

$$
u_i(x)=\sum_{R\in\mathcal F:\ i\in R}x_R.
$$

An integer vector $u$ is fractionally feasible if some such $x$ satisfies
$u_i(x)\ge u_i$ for every voter. It is integrally feasible if this can be
done with $x_R\in\{0,1\}$. A fractionally but not integrally feasible
instance-utility pair is a **hole**.

The candidate minimal holes in the paper satisfy:

- **(R1)** $\mathcal F$ is an antichain;
- **(R2)** all residual supplies are one, with no singleton or full-voter type;
- **(R3)** $2\le d_i\le |\mathcal F|-1$ for every voter;
- **(R4)** $2\le k\le|\mathcal F|-2$ and $|\mathcal F|\le n$;
- **(R5)** $1\le u_i\le\min\{d_i-1,k-1\}$.

For seven voters, **(R6)** requires $\bigcap_{R\in\mathcal F}R=\varnothing$.
The resulting 54,985 holes are then filtered by the Lindahl-compatibility
condition **(R7)**: there is a $\beta\in(0,1]^7$ with
$\sum_{i\in R}\beta_i=1$ for every $R\in\mathcal F$.


## 2. Configuration

The default is a complete six-voter rebuild. For the full seven-voter run,
set `N = 7` and leave `REBUILD_CATALOGUE` true; the recorded four-worker run
took about 2.5 hours. Setting `REBUILD_CATALOGUE` false performs only the
fact-property checks on the bundled precomputed catalogue.

The values may also be supplied through `CORE67_N`, `CORE67_JOBS`,
`CORE67_REBUILD`, and `CORE67_OUTPUT` for headless execution.


In [1]:
from __future__ import annotations

import gzip
import io
import itertools
import json
import os
import subprocess
import time
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Iterable, Sequence

from IPython.display import Markdown, display
from joblib import Parallel, delayed
from z3 import Q, Real, Solver, Sum, sat, simplify, unsat

N = int(os.environ.get("CORE67_N", "6"))
JOBS = int(os.environ.get("CORE67_JOBS", str(max(1, min(4, os.cpu_count() or 1)))))
REBUILD_CATALOGUE = os.environ.get("CORE67_REBUILD", "1").lower() not in {"0", "false", "no"}

ROOT = Path.cwd()
if not (ROOT / "enumerate_antichains.c").exists():
    ROOT = ROOT / "core67-repro"
if not (ROOT / "enumerate_antichains.c").exists():
    raise FileNotFoundError("Run the notebook from core67-repro/ or the repository root.")

OUTPUT = Path(os.environ.get("CORE67_OUTPUT", str(ROOT / f"notebook-results-n{N}")))
SAVED_CATALOGUE = ROOT / f"results-n{N}" / f"holes{N}.json.gz"
OUTPUT.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    6: {"holes": 50, "families": 23},
    7: {
        "holes": 54_985,
        "families": 10_292,
        "lindahl_compatible": 21_818,
        "class_1": 21_520,
        "class_2": 298,
    },
}
if N not in EXPECTED:
    raise ValueError("This reproduction treats N=6 and N=7 only.")
if not REBUILD_CATALOGUE and not SAVED_CATALOGUE.exists():
    raise FileNotFoundError(f"No bundled catalogue at {SAVED_CATALOGUE}")

run_started = time.time()
print(f"N={N}; jobs={JOBS}; rebuild_catalogue={REBUILD_CATALOGUE}")
print(f"output directory: {OUTPUT}")


N=6; jobs=4; rebuild_catalogue=True
output directory: /Users/dominik/GitHub/balanced-abc/core67-repro/notebook-results-n6


## 3. Candidate types and integral committees

A subset of voters is represented by a bit mask. The helpers below translate
that encoding and enumerate all utilities of size-$k$ integral committees.
It suffices to use exactly $k$ candidates: by (R4), a smaller committee can
always be padded without decreasing utility.


In [2]:
def popcount(mask: int) -> int:
    return mask.bit_count()


def bit_string(mask: int, n: int) -> str:
    return "".join("1" if mask >> i & 1 else "0" for i in range(n))


def dominates(a: Sequence[int], b: Sequence[int]) -> bool:
    return all(x >= y for x, y in zip(a, b))


def committee_utilities(types: Sequence[int], k: int, n: int) -> list[tuple[int, ...]]:
    """Utilities of all size-k committees.

    It suffices to use exactly k candidates: any smaller committee can be
    padded with unused candidates without decreasing utilities.
    """
    result = []
    for committee in itertools.combinations(types, k):
        result.append(tuple(sum(mask >> i & 1 for mask in committee) for i in range(n)))
    return result

def mask_from_voters(voters: str) -> int:
    return sum(1 << (int(voter) - 1) for voter in voters)


def voter_set(mask: int, n: int) -> str:
    return "".join(str(i + 1) for i in range(n) if mask >> i & 1)


### Example: two disjoint triangles

The first six-voter hole in the paper is

$$
\mathcal F=\{12,13,23,45,46,56\},\quad k=3,\quad
u=(1,1,1,1,1,1).
$$

Putting $x_R=1/2$ on every type uses three seats and gives every voter
utility one. The following exhaustive check confirms that none of the 20
integral committees of size three weakly dominates $u$.


In [3]:
triangle_types = tuple(mask_from_voters(label) for label in ("12", "13", "23", "45", "46", "56"))
triangle_u = (1, 1, 1, 1, 1, 1)
triangle_x = [Q(1, 2)] * 6

fractional_loads = [
    sum((triangle_x[j] for j, mask in enumerate(triangle_types) if mask >> i & 1), Q(0, 1))
    for i in range(6)
]
integral_vectors = committee_utilities(triangle_types, 3, 6)
print("fractional committee size:", simplify(sum(triangle_x, Q(0, 1))))
print("fractional voter utilities:", [simplify(load) for load in fractional_loads])
print("integral committees checked:", len(integral_vectors))
print("integrally feasible:", any(dominates(vector, triangle_u) for vector in integral_vectors))


fractional committee size: 3
fractional voter utilities: [1, 1, 1, 1, 1, 1]
integral committees checked: 20
integrally feasible: False


## 4. Enumerating structural candidates

The C helper constructs (R1) antichains directly, filters the structural
conditions (R2)–(R4) and (R6), applies a safe prefilter derived from (R5),
and keeps one lexicographically least representative under voter relabeling.
Its root branches can be split between workers. This is purely combinatorial;
no feasibility solver is used yet.


In [4]:
def generate_candidates(n: int, jobs: int, output: Path) -> tuple[list[tuple[int, ...]], float]:
    start = time.time()
    source = ROOT / "enumerate_antichains.c"
    binary = output / f"enumerate-n{n}"
    subprocess.run(
        [os.environ.get("CC", "cc"), "-O3", "-std=c11", f"-DNV={n}", str(source), "-o", str(binary)],
        check=True,
    )
    processes = []
    handles = []
    for worker in range(jobs):
        candidates = output / f"candidates-{worker}.txt"
        log = (output / f"enumerator-{worker}.txt").open("w", encoding="utf-8")
        handles.append(log)
        processes.append(subprocess.Popen(
            [str(binary), str(candidates), str(worker), str(jobs)], stdout=log, stderr=subprocess.STDOUT
        ))
    for process in processes:
        if process.wait() != 0:
            raise RuntimeError("antichain generator failed; see enumerator logs")
    for handle in handles:
        handle.close()

    lines = set()
    for worker in range(jobs):
        lines.update((output / f"candidates-{worker}.txt").read_text(encoding="utf-8").splitlines())
    families = sorted(tuple(map(int, line.split())) for line in lines if line.strip())
    (output / "candidates.txt").write_text(
        "".join(" ".join(map(str, family)) + "\n" for family in families), encoding="utf-8"
    )
    elapsed = time.time() - start
    print(f"antichain enumeration: {len(families):,} candidates in {elapsed:.1f}s", flush=True)
    return families, elapsed

if REBUILD_CATALOGUE:
    families, enumeration_seconds = generate_candidates(N, JOBS, OUTPUT)
    print(f"canonical candidate families: {len(families):,}")
    print(f"enumeration time: {enumeration_seconds:.3f} seconds")
    for family in families[:3]:
        print([voter_set(mask, N) for mask in family])
else:
    families, enumeration_seconds = [], None
    print("Skipped catalogue construction: using the bundled precomputed catalogue.")


antichain enumeration: 1,023 candidates in 0.7s


canonical candidate families: 1,023
enumeration time: 0.656 seconds
['12', '13', '23', '14', '156', '456']
['12', '13', '23', '14', '256', '456']
['12', '13', '23', '14', '256', '3456']


## 5. Exact fractional feasibility

For fixed $(\mathcal F,k,u)$, Z3 checks

$$
0\le x_R\le1,\qquad \sum_Rx_R\le k,\qquad
\sum_{R\ni i}x_R\ge u_i\quad(i\in N).
$$

These exact rational linear constraints are precisely the definition of
fractional-committee feasibility used for candidate minimal holes.


In [5]:
def witness_solver(types: Sequence[int], k: int, utility: Sequence[int]) -> tuple[Solver, list]:
    x = [Real(f"x_{j}") for j in range(len(types))]
    solver = Solver()
    for value in x:
        solver.add(value >= 0, value <= 1)
    solver.add(Sum(x) <= k)
    for i, target in enumerate(utility):
        solver.add(Sum([x[j] for j, mask in enumerate(types) if mask >> i & 1]) >= target)
    return solver, x


def fractionally_feasible(types: Sequence[int], k: int, utility: Sequence[int]) -> bool:
    solver, _ = witness_solver(types, k, utility)
    return solver.check() == sat

solver, variables = witness_solver(triangle_types, 3, triangle_u)
assert solver.check() == sat
model = solver.model()
print("one exact witness:", [model.eval(variable) for variable in variables])


one exact witness: [1/2, 1/2, 1/2, 1/2, 1/2, 1/2]


## 6. Symmetry of utility vectors

The C program has already quotiented type families by $S_N$. A fixed family
can still have automorphisms, so utility vectors related by one of those
automorphisms are the same hole. We retain their lexicographically least
representative.


In [6]:
@lru_cache(maxsize=2)
def permutation_data(n: int) -> tuple[tuple[tuple[int, ...], ...], tuple[tuple[int, ...], ...]]:
    permutations = tuple(itertools.permutations(range(n)))
    tables = []
    for permutation in permutations:
        table = []
        for mask in range(1 << n):
            image = sum(1 << permutation[i] for i in range(n) if mask >> i & 1)
            table.append(image)
        tables.append(tuple(table))
    return permutations, tuple(tables)


def canonical_utilities(types: tuple[int, ...], utilities: Iterable[tuple[int, ...]], n: int):
    """Quotient utilities by automorphisms of an already-canonical family."""
    permutations, tables = permutation_data(n)
    automorphisms = [
        permutation
        for permutation, table in zip(permutations, tables)
        if tuple(sorted(table[mask] for mask in types)) == types
    ]
    result = set()
    for utility in utilities:
        images = []
        for permutation in automorphisms:
            image = [0] * n
            for old, new in enumerate(permutation):
                image[new] = utility[old]
            images.append(tuple(image))
        result.add(min(images))
    return sorted(result)


## 7. Testing every permitted utility vector

For each $2\le k\le|\mathcal F|-2$, condition (R5) bounds the utility box.
The next function first rejects integrally feasible vectors by exhaustive
committee enumeration, uses a safe total-utility shortcut, and invokes Z3
only on the survivors. Symmetric utilities are collapsed at the end.


In [7]:
def verify_family(arguments) -> list[dict]:
    """Enumerate the holes supported by one canonical antichain.

    Integral infeasibility and the elementary total-utility bound are checked
    first.  Z3 is called only for the surviving utility vectors.
    """
    n, types = arguments
    types = tuple(types)
    degrees = [sum(mask >> i & 1 for mask in types) for i in range(n)]
    sizes = sorted((popcount(mask) for mask in types), reverse=True)
    raw_holes: list[tuple[int, tuple[int, ...]]] = []

    for k in range(2, len(types) - 1):
        upper = tuple(min(degrees[i], k) - 1 for i in range(n))
        integral = committee_utilities(types, k, n)
        if any(dominates(vector, upper) for vector in integral):
            continue
        size_bound = sum(sizes[:k])
        for utility in itertools.product(*(range(1, bound + 1) for bound in upper)):
            if sum(utility) > size_bound:
                continue
            if any(dominates(vector, utility) for vector in integral):
                continue
            if fractionally_feasible(types, k, utility):
                raw_holes.append((k, utility))

    if not raw_holes:
        return []
    by_k: dict[int, list[tuple[int, ...]]] = defaultdict(list)
    for k, utility in raw_holes:
        by_k[k].append(utility)
    result = []
    for k, utilities in by_k.items():
        for utility in canonical_utilities(types, utilities, n):
            result.append({
                "types": [bit_string(mask, n) for mask in types],
                "masks": list(types),
                "k": k,
                "u": list(utility),
            })
    return result


## 8. Building or loading the hole catalogue

Joblib serializes notebook-defined workers reliably. The gzip JSON writer fixes
the timestamp, making the compressed catalogue deterministic. This cell is the
dominant part of the full seven-voter reproduction.


In [8]:
def load_catalogue(path: Path) -> list[dict]:
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as stream:
        records = json.load(stream)
    for record in records:
        if "masks" not in record:
            record["masks"] = [
                sum(1 << i for i, bit in enumerate(bits) if bit == "1") for bits in record["types"]
            ]
    return records


def write_gzip_json(path: Path, value) -> None:
    with path.open("wb") as raw:
        with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8") as text:
                json.dump(value, text, separators=(",", ":"), sort_keys=True)

if REBUILD_CATALOGUE:
    verification_started = time.time()
    batch_size = max(1, len(families) // max(1, JOBS * 64))
    chunks = Parallel(n_jobs=JOBS, backend="loky", batch_size=batch_size)(
        delayed(verify_family)((N, family)) for family in families
    )
    records = [record for chunk in chunks for record in chunk]
    records.sort(key=lambda record: (record["masks"], record["k"], record["u"]))
    catalogue_path = OUTPUT / f"holes{N}.json.gz"
    write_gzip_json(catalogue_path, records)
    hole_verification_seconds = time.time() - verification_started
    print(f"holes: {len(records):,}")
    print(f"candidate-to-hole verification: {hole_verification_seconds:.3f} seconds")
else:
    catalogue_path = SAVED_CATALOGUE
    records = load_catalogue(catalogue_path)
    hole_verification_seconds = None
    print(f"loaded {len(records):,} records from {catalogue_path}")


holes: 50
candidate-to-hole verification: 5.504 seconds


### Example catalogue records

The stored bit strings are ordered by voter number: `110000` denotes type
$12$. The display converts the first five records back to the notation used
in the paper.


In [9]:
def readable_record(record: dict, n: int) -> dict:
    return {
        "types": [voter_set(mask, n) for mask in record["masks"]],
        "k": record["k"],
        "u": record["u"],
    }

for index, record in enumerate(records[:5], 1):
    print(f"{index}: {readable_record(record, N)}")


1: {'types': ['12', '13', '23', '45', '46', '56'], 'k': 3, 'u': [1, 1, 1, 1, 1, 1]}
2: {'types': ['12', '13', '234', '235', '236', '456'], 'k': 3, 'u': [1, 2, 2, 1, 1, 1]}
3: {'types': ['12', '34', '135', '245', '236', '146'], 'k': 2, 'u': [1, 1, 1, 1, 1, 1]}
4: {'types': ['12', '34', '135', '245', '236', '146'], 'k': 3, 'u': [1, 1, 2, 2, 1, 1]}
5: {'types': ['12', '134', '135', '245', '236', '146'], 'k': 2, 'u': [1, 1, 1, 1, 1, 1]}


## 9. The two utility properties

The facts use two properties of a voter $i$: $u-e_i$ is integrally feasible,
and every fractional witness gives exactly utility $u_i$. Since feasibility
already forces $u_i(x)\ge u_i$, the second property is proved by showing that
$u_i(x)>u_i$ is unsatisfiable.

The six-voter fact requires both properties for every voter of all 50 holes.
Class (1) of the seven-voter fact requires a voter with both properties.


In [10]:
def integrally_reducible_voters(types: Sequence[int], k: int, utility: Sequence[int], n: int) -> list[int]:
    integral = committee_utilities(types, k, n)
    result = []
    for i in range(n):
        target = list(utility)
        target[i] -= 1
        if any(dominates(vector, target) for vector in integral):
            result.append(i)
    return result


def exact_witness_voters(types: Sequence[int], k: int, utility: Sequence[int], n: int) -> list[int]:
    solver, x = witness_solver(types, k, utility)
    if solver.check() != sat:
        raise AssertionError("catalogue contains a fractionally infeasible entry")
    result = []
    for i in range(n):
        # Feasibility already forces load_i >= utility_i.  Consequently the
        # strict counterexample below is unsatisfiable exactly when every
        # witness gives voter i the integer utility utility_i.
        load = Sum([x[j] for j, mask in enumerate(types) if mask >> i & 1])
        solver.push()
        solver.add(load > utility[i])
        verdict = solver.check()
        solver.pop()
        if verdict == unsat:
            result.append(i)
    return result

example = records[0]
types = tuple(example["masks"])
print("example:", readable_record(example, N))
print("u-e_i integrally feasible:", [
    i + 1 for i in integrally_reducible_voters(types, example["k"], example["u"], N)
])
print("exact utility in every witness:", [
    i + 1 for i in exact_witness_voters(types, example["k"], example["u"], N)
])


example: {'types': ['12', '13', '23', '45', '46', '56'], 'k': 3, 'u': [1, 1, 1, 1, 1, 1]}
u-e_i integrally feasible: [1, 2, 3, 4, 5, 6]
exact utility in every witness: [1, 2, 3, 4, 5, 6]


## 10. Lindahl-compatible seven-voter holes: condition (R7)

Condition (R7) requires a strictly positive vector

$$
\beta\in(0,1]^7,\qquad \sum_{i\in R}\beta_i=1
\quad(R\in\mathcal F).
$$

This depends only on the type family. The closed polytope used for the final
bound replaces strict positivity by $\beta_i\ge0$.


In [11]:
def beta_solver(types: Sequence[int], n: int, *, positive: bool) -> tuple[Solver, list]:
    beta = [Real(f"beta_{i}") for i in range(n)]
    solver = Solver()
    for value in beta:
        solver.add(value > 0 if positive else value >= 0, value <= 1)
    for mask in types:
        solver.add(Sum([beta[i] for i in range(n) if mask >> i & 1]) == 1)
    return solver, beta


def analyze_family(arguments) -> dict:
    """Analyze all holes on one family, sharing its (R7) feasibility check."""
    n, records = arguments
    types = tuple(records[0]["masks"])
    satisfies_r7 = True
    if n == 7:
        solver, _ = beta_solver(types, n, positive=True)
        satisfies_r7 = solver.check() == sat
    result = {
        "holes": len(records),
        "lindahl_compatible": len(records) if n == 7 and satisfies_r7 else 0,
        "checked": [],
    }

    for record in records:
        utility, k = record["u"], record["k"]
        entry = {}
        if n == 6 or satisfies_r7:
            reducible = integrally_reducible_voters(types, k, utility, n)
            exact = exact_witness_voters(types, k, utility, n)
            entry.update({
                "record": record,
                "integrally_reducible": reducible,
                "exact_witness": exact,
                "class_1": bool(set(reducible) & set(exact)),
            })
        if entry:
            result["checked"].append(entry)
    return result


## 11. Class (2): the $8/9$ bound

For each of the 298 Lindahl-compatible holes outside class (1), and for every
voter $i$, the paper checks

$$
\max_{\beta\in\Delta^{\mathcal F}}
\left(\beta_i+k-\sum_j u_j\beta_j\right)\le\frac89<1.
$$

Z3 is asked whether a feasible point with value strictly greater than $8/9$
exists. Unsatisfiability proves the bound exactly.


In [12]:
def check_class_2_bound(class_2: Sequence[dict], n: int) -> dict:
    """Prove the 8/9 bound by exact counterexample queries."""
    bound = Q(8, 9)
    for entry in class_2:
        record = entry["record"]
        types, utility, k = record["masks"], record["u"], record["k"]
        solver, beta = beta_solver(types, n, positive=False)
        for i in range(n):
            expression = beta[i] + k - Sum([utility[j] * beta[j] for j in range(n)])
            solver.push()
            solver.add(expression > bound)
            verdict = solver.check()
            solver.pop()
            if verdict != unsat:
                raise AssertionError(f"class (2) bound fails for {record}, voter {i + 1}")
    return {"bound": "8/9"}


## 12. Run the fact checks

We group holes by family, inspect every six-voter hole and every
Lindahl-compatible seven-voter hole, and then assert all paper counts. These
assertions cannot be disabled in a reproduction run.


In [13]:
def check_expected(summary: dict) -> None:
    expected = EXPECTED[summary["n"]]
    for key, value in expected.items():
        if summary.get(key) != value:
            raise AssertionError(f"expected {key}={value:,}, obtained {summary.get(key)!r}")
    if summary["n"] == 6:
        if not summary["all_voters_exact_in_every_witness"]:
            raise AssertionError("a six-voter witness gives some voter utility above u_i")
        if not summary["all_reduced_utilities_integrally_feasible"]:
            raise AssertionError("some six-voter vector u-e_i is not integrally feasible")
    else:
        if not summary["class_2_reducible_at_every_voter"]:
            raise AssertionError("some class (2) vector u-e_i is not integrally feasible")

groups: dict[tuple[int, ...], list[dict]] = defaultdict(list)
for record in records:
    groups[tuple(record["masks"])].append(record)

analysis_started = time.time()
arguments = [(N, group) for _, group in sorted(groups.items())]
batch_size = max(1, len(arguments) // max(1, JOBS * 32))
family_results = Parallel(n_jobs=JOBS, backend="loky", batch_size=batch_size)(
    delayed(analyze_family)(argument) for argument in arguments
)
checked = [entry for result in family_results for entry in result["checked"]]
lindahl_compatible = sum(result["lindahl_compatible"] for result in family_results)

summary = {"n": N, "holes": len(records), "families": len(groups)}
class_2 = []
if N == 6:
    summary.update({
        "all_voters_exact_in_every_witness": all(
            len(entry["exact_witness"]) == N for entry in checked
        ),
        "all_reduced_utilities_integrally_feasible": all(
            len(entry["integrally_reducible"]) == N for entry in checked
        ),
    })
else:
    class_2 = [entry for entry in checked if not entry["class_1"]]
    summary.update({
        "lindahl_compatible": lindahl_compatible,
        "class_1": len(checked) - len(class_2),
        "class_2": len(class_2),
        "class_2_reducible_at_every_voter": all(
            len(entry["integrally_reducible"]) == N for entry in class_2
        ),
        "class_2_bound": check_class_2_bound(class_2, N),
    })

summary["elapsed_seconds"] = round(time.time() - run_started, 3)
summary["timing"] = {"fact_check_seconds": round(time.time() - analysis_started, 3)}
if REBUILD_CATALOGUE:
    summary["timing"].update({
        "candidate_antichains": len(families),
        "antichain_enumeration_seconds": round(enumeration_seconds, 3),
        "hole_verification_seconds": round(hole_verification_seconds, 3),
        "catalogue_total_seconds": round(enumeration_seconds + hole_verification_seconds, 3),
    })
summary["z3_version"] = __import__("z3").get_version_string()

check_expected(summary)
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
if class_2:
    write_gzip_json(OUTPUT / "class-2.json.gz", class_2)

display(Markdown("### Verified summary"))
print(json.dumps(summary, indent=2, sort_keys=True))


### Verified summary

{
  "all_reduced_utilities_integrally_feasible": true,
  "all_voters_exact_in_every_witness": true,
  "elapsed_seconds": 6.381,
  "families": 23,
  "holes": 50,
  "n": 6,
  "timing": {
    "antichain_enumeration_seconds": 0.656,
    "candidate_antichains": 1023,
    "catalogue_total_seconds": 6.16,
    "hole_verification_seconds": 5.504,
    "fact_check_seconds": 0.092
  },
  "z3_version": "4.15.3"
}


## 13. How to read success

For $n=6$: 50 holes on 23 families; for every voter of every hole, every
fractional witness gives utility exactly $u_i$, and $u-e_i$ is integrally feasible.

For $n=7$: 54,985 holes on 10,292 families; 21,818 satisfy (R7); 21,520
belong to class (1); 298 belong to class (2), where $u-e_i$ is integrally
feasible for every voter and the closed-$\beta$ expression is at most $8/9$.

The bundled complete seven-voter result is displayed below as a reference even
when the notebook itself is run for six voters.


In [14]:
reference_path = ROOT / "results-n7" / "summary.json"
if reference_path.exists():
    reference = json.loads(reference_path.read_text(encoding="utf-8"))
    rows = [
        ("holes", reference["holes"]),
        ("antichain families", reference["families"]),
        ("holes satisfying (R7)", reference["lindahl_compatible"]),
        ("class (1)", reference["class_1"]),
        ("class (2)", reference["class_2"]),
        ("class (2) bound", reference["class_2_bound"]["bound"]),
    ]
    table = "\n".join(
        f"| {label} | {value:,} |" if isinstance(value, int) else f"| {label} | {value} |"
        for label, value in rows
    )
    display(Markdown("### Bundled seven-voter reference\n\n| quantity | value |\n|---|---:|\n" + table))


### Bundled seven-voter reference

| quantity | value |
|---|---:|
| holes | 54,985 |
| antichain families | 10,292 |
| holes satisfying (R7) | 21,818 |
| class (1) | 21,520 |
| class (2) | 298 |
| class (2) bound | 8/9 |

## 14. Conclusion

In rebuild mode, the notebook reconstructs the finite search, checks fractional
feasibility and integral infeasibility, and removes relabeling duplicates. In
saved-catalogue mode, it loads that precomputed search result. For six voters,
it checks the two utility properties. For seven voters, it checks condition
(R7), the two classes, and the $8/9$ bound stated in the paper.
